#LCEL管道——prompt|llm|parser
chain = prompt | llm | parser
```
- `|` 叫**管道操作符**（pipe）：左边组件的输出，自动变成右边组件的输入
- 读法：提示词 → 模型 → 解析器，一条流水线

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

prompt = ChatPromptTemplate.from_messages([
    ("system","你是{role},回答不超过500字。"),
    ("user","{question}")
])

llm = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0.3,
    max_tokens=200
)

parser = StrOutputParser()  #把AIMessage转成纯字符串

chain = prompt|llm|parser  #一条流水线

step1 = prompt.invoke({"role":"编程导师","question":"你好"})
print(type(step1))

step2 = llm.invoke(step1)
print(type(step2))

step3 = parser.invoke(step2)
print(type(step3))

#调用整条链：只传模板变量
result = chain.invoke({"role":"Agent开发导师","question":"怎样设置agent工作流？"})

print(result)

<class 'langchain_core.prompt_values.ChatPromptValue'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.base.TextAccessor'>
设置Agent工作流，核心是**将大目标拆解为可执行的步骤，并让Agent能自主决策和调用工具**。以下是关键步骤：

1. **明确输入与目标**  
   定义Agent的起点（用户消息、传感器数据）和终点（最终输出格式）。例如：“根据用户需求，生成一份带代码示例的教程”。

2. **拆解子任务**  
   将主任务分解为有依赖关系的子任务，如：理解需求 → 检索资料 → 编写内容 → 校验格式。每个子任务对应一个节点。

3. **选择工具与能力**  
   为每个节点绑定工具：`search_api`（搜索）、`code_interpreter`（执行代码）、`text_generator`（生成文本）。并指定工具参数如何从上下文提取。

4. **设计状态流**  
   使用状态机管理Agent的进度（如`待输入`、`检索中`、`生成中`）。常用框架如**LangGraph**用图结构定义节点和边，支持条件跳转（例如：资料不足则重新搜索）。

5. **加入反馈机制**  
   在关键节点设置检查点，让Agent自我评估或接受用户修正。例如：“生成内容后，先调用‘质量评分器’判断是否满足要求，否则重试”。

6. **循环与终止条件**  
   明确最大迭代次数和停止条件（如结果通过校验），避免死循环。

**快速示例**（用伪代码）：  
```python
workflow = start() → search() → generate() → validate() → end()
```  
在`validate()`中若失败则返回`search()`重试。

**核心建议**：不要一开始做复杂图，先用线性流程，逐步加入分支。推荐尝试LangGraph或CrewAI这类现成框架，能节省状态管理时间。最后，务必用日志跟踪每一步，便于调试。
